# ECS Generation - Self-Validating Testing

This notebook implements **self-validating tests** for the ECS Generation module.

## 1. Setup and Imports

In [1]:
# Core imports
import json
import pandas as pd
import numpy as np
from datetime import datetime
from typing import Dict, List, Any, Optional, Tuple
from dataclasses import dataclass, asdict, field
import uuid
import traceback
import time
import re

# DataIku imports
import dataiku
import dataikuapi

# Fuzzy matching
from rapidfuzz import fuzz

# Project imports
from utils import connection
from utilities.variables import RD_PROJECT_NAME, SECRET_NAME, TOKEN_KEY
from utilities.logging_config import logging

print(f"Self-Validating Test Session: {datetime.now().isoformat()}")

Self-Validating Test Session: 2026-03-17T18:48:46.892406


In [2]:
# Initialize DataIku connection
DATAIKU_HOST, API_SECRET_KEY = connection.get_dataiku_host_and_api_key(
    RD_PROJECT_NAME, SECRET_NAME, TOKEN_KEY
)

client = dataikuapi.DSSClient(DATAIKU_HOST, API_SECRET_KEY)
project = client.get_project(RD_PROJECT_NAME)

print(f"Connected to project: {RD_PROJECT_NAME}")

Connected to project: ECSGENERATION


## 2. Data Classes and Constants

In [3]:
@dataclass
class MetricResult:
    """Single metric evaluation result."""
    name: str
    score: float
    passed: bool
    threshold: float
    details: Dict[str, Any]
    category: str = ""


@dataclass
class EvaluationResult:
    """Complete evaluation result for a single generation."""
    eval_id: str
    timestamp: datetime
    input_form_name: str
    input_field_count: int
    output_rule_count: int
    metrics: Dict[str, MetricResult]
    overall_score: float
    passed: bool
    duration_ms: int
    metadata: Dict[str, Any]


# Valid CDISC domain codes
VALID_CDISC_DOMAINS = {
    "AE", "BE", "CE", "CM", "CO", "CV", "DA", "DD", "DM", "DS", 
    "DV", "EC", "EG", "EX", "FA", "FT", "HO", "IE", "IS", "LB", 
    "MB", "MH", "MI", "MK", "ML", "MO", "MS", "NV", "OE", "PC", 
    "PE", "PP", "PR", "QS", "RE", "RP", "RS", "SC", "SE", "SG",
    "SM", "SR", "SS", "SU", "SV", "TA", "TD", "TE", "TI", "TM",
    "TR", "TS", "TU", "TV", "UR", "VS", "XA"
}

# Required keys in ECS output
REQUIRED_OUTPUT_KEYS = [
    "form_name", 
    "form_field_value", 
    "validation_logic", 
    "form_domain_name"
]

OPTIONAL_OUTPUT_KEYS = [
    "ecs_id",
    "source",
    "field_oids"
]

# Thresholds for self-validating metrics
SELF_VALIDATION_THRESHOLDS = {
    # Schema & Completeness
    "schema_validity": 0.95,
    "field_coverage": 0.90,
    "non_empty_logic": 0.95,
    "unique_ecs_ids": 1.0,
    
    # Echo-Back Consistency
    "form_name_echo": 0.80,
    "field_name_echo": 0.75,
    "domain_validity": 0.90,
    
    # Semantic Quality (LLM-as-Judge)
    "validation_relevance": 0.70,
    "logic_coherence": 0.80,
    "clinical_plausibility": 0.70,
    
    # Regression & Performance
    "determinism_score": 0.80,
    "latency_check": 0.90,
    "token_efficiency": 0.90,
}

# Performance baselines
LATENCY_BASELINE_MS = 30000  # 30 seconds per generation
TOKENS_PER_FIELD_BASELINE = 150  # Expected tokens per field

print("Data classes and constants initialized")
print(f"Valid CDISC domains: {len(VALID_CDISC_DOMAINS)}")
print(f"Thresholds defined: {len(SELF_VALIDATION_THRESHOLDS)}")

Data classes and constants initialized
Valid CDISC domains: 57
Thresholds defined: 13


## 3. Self-Validating Evaluator

In [4]:
class SelfValidatingEvaluator:
    """
    Evaluates ECS generation WITHOUT requiring ground truth.
    Can run on EVERY generation request.
    
    Metrics Categories:
    1. Schema & Completeness - structural validation
    2. Echo-Back Consistency - input/output alignment
    3. Semantic Quality - LLM-judged content quality
    4. Regression & Performance - stability checks
    """
    
    def __init__(self, client, project, thresholds: Dict[str, float] = None,
                 enable_llm_judge: bool = True):
        self.client = client
        self.project = project
        self.thresholds = thresholds or SELF_VALIDATION_THRESHOLDS
        self.enable_llm_judge = enable_llm_judge
        self.results_history = []
        self._init_llm_judge()
    
    def _init_llm_judge(self):
        """Initialize LLM for semantic quality judging."""
        if not self.enable_llm_judge:
            self.judge_llm = None
            return
        try:
            # Use the same pattern as ecs_generation_agent.py - as_langchain_llm()
            # This avoids the ArrayIndexOutOfBoundsException from with_message()
            llm_model = self.project.get_variables().get('local', {}).get(
                'default_llm_model', 'openai:gpt-4o-mini'
            )
            self.judge_llm = self.project.get_llm(llm_model).as_langchain_llm(
                completion_settings={
                    "temperature": 0,
                    "timeout": 60,
                    "max_tokens": 100  # Small response for scoring
                }
            )
            print(f"LLM Judge initialized ({llm_model})")
        except Exception as e:
            print(f"LLM Judge not available: {e}")
            self.judge_llm = None
            self.enable_llm_judge = False
    
    # ============ SCHEMA & COMPLETENESS ============
    
    def check_schema_validity(self, output: List[Dict]) -> MetricResult:
        """
        1.1 Schema Validity
        Checks if all required keys are present in output.
        """
        if not output:
            return MetricResult(
                name="schema_validity",
                score=0,
                passed=False,
                threshold=self.thresholds["schema_validity"],
                details={"error": "Empty output"},
                category="Schema & Completeness"
            )
        
        valid_count = 0
        missing_keys_sample = []
        
        for i, item in enumerate(output):
            missing = [k for k in REQUIRED_OUTPUT_KEYS if k not in item or not item[k]]
            if not missing:
                valid_count += 1
            elif len(missing_keys_sample) < 3:
                missing_keys_sample.append({"index": i, "missing": missing})
        
        score = valid_count / len(output)
        
        return MetricResult(
            name="schema_validity",
            score=score,
            passed=score >= self.thresholds["schema_validity"],
            threshold=self.thresholds["schema_validity"],
            details={
                "valid_count": valid_count,
                "total_count": len(output),
                "required_keys": REQUIRED_OUTPUT_KEYS,
                "missing_keys_sample": missing_keys_sample
            },
            category="Schema & Completeness"
        )
    
    def check_field_coverage(self, input_fields: List[Dict], 
                             output: List[Dict]) -> MetricResult:
        """
        1.2 Field Coverage
        Every input field should have a corresponding output rule.
        """
        input_names = set(
            f.get("form_field_value", f.get("field_name", "")).lower().strip()
            for f in input_fields
        )
        input_names.discard("")
        
        output_names = set(
            r.get("form_field_value", "").lower().strip()
            for r in output
        )
        output_names.discard("")
        
        if not input_names:
            return MetricResult(
                name="field_coverage",
                score=1.0,
                passed=True,
                threshold=self.thresholds["field_coverage"],
                details={"note": "No input fields to check"},
                category="Schema & Completeness"
            )
        
        # Use fuzzy matching to find covered fields
        covered = set()
        for inp in input_names:
            for out in output_names:
                if fuzz.ratio(inp, out) >= 80:
                    covered.add(inp)
                    break
        
        missing = input_names - covered
        score = len(covered) / len(input_names)
        
        return MetricResult(
            name="field_coverage",
            score=score,
            passed=score >= self.thresholds["field_coverage"],
            threshold=self.thresholds["field_coverage"],
            details={
                "input_count": len(input_names),
                "output_count": len(output_names),
                "covered_count": len(covered),
                "missing_fields": list(missing)[:5]
            },
            category="Schema & Completeness"
        )
    
    def check_non_empty_logic(self, output: List[Dict]) -> MetricResult:
        """
        1.3 Non-Empty Logic
        Validation logic should not be empty or trivial.
        """
        if not output:
            return MetricResult(
                name="non_empty_logic",
                score=0,
                passed=False,
                threshold=self.thresholds["non_empty_logic"],
                details={"error": "Empty output"},
                category="Schema & Completeness"
            )
        
        non_empty = 0
        empty_samples = []
        
        for i, item in enumerate(output):
            logic = str(item.get("validation_logic", "")).strip()
            # Check for meaningful content (more than just whitespace or placeholders)
            if logic and len(logic) > 10 and logic.lower() not in ["n/a", "none", "null", "tbd"]:
                non_empty += 1
            elif len(empty_samples) < 3:
                empty_samples.append({
                    "index": i,
                    "field": item.get("form_field_value", "unknown"),
                    "logic": logic[:50] if logic else "<empty>"
                })
        
        score = non_empty / len(output)
        
        return MetricResult(
            name="non_empty_logic",
            score=score,
            passed=score >= self.thresholds["non_empty_logic"],
            threshold=self.thresholds["non_empty_logic"],
            details={
                "non_empty_count": non_empty,
                "total_count": len(output),
                "empty_samples": empty_samples
            },
            category="Schema & Completeness"
        )
    
    def check_unique_ecs_ids(self, output: List[Dict]) -> MetricResult:
        """
        1.4 Unique ECS IDs
        Each rule should have a unique identifier.
        """
        ecs_ids = [r.get("ecs_id", "") for r in output if r.get("ecs_id")]
        
        if not ecs_ids:
            # No ECS IDs generated - not a failure, just a note
            return MetricResult(
                name="unique_ecs_ids",
                score=1.0,
                passed=True,
                threshold=self.thresholds["unique_ecs_ids"],
                details={"note": "No ECS IDs in output - skipped"},
                category="Schema & Completeness"
            )
        
        unique_ids = set(ecs_ids)
        duplicates = [id for id in unique_ids if ecs_ids.count(id) > 1]
        score = len(unique_ids) / len(ecs_ids)
        
        return MetricResult(
            name="unique_ecs_ids",
            score=score,
            passed=score >= self.thresholds["unique_ecs_ids"],
            threshold=self.thresholds["unique_ecs_ids"],
            details={
                "total_ids": len(ecs_ids),
                "unique_ids": len(unique_ids),
                "duplicates": duplicates[:5]
            },
            category="Schema & Completeness"
        )
    
    # ============ ECHO-BACK CONSISTENCY ============
    
    def check_form_name_echo(self, input_form_name: str, 
                              output: List[Dict]) -> MetricResult:
        """
        2.1 Form Name Echo
        Output form names should match the input form name.
        """
        if not output or not input_form_name:
            return MetricResult(
                name="form_name_echo",
                score=0,
                passed=False,
                threshold=self.thresholds["form_name_echo"],
                details={"error": "Missing input or output"},
                category="Echo-Back Consistency"
            )
        
        input_lower = input_form_name.lower().strip()
        scores = []
        mismatches = []
        
        for i, item in enumerate(output):
            output_form = str(item.get("form_name", "")).lower().strip()
            similarity = fuzz.ratio(input_lower, output_form) / 100
            scores.append(similarity)
            if similarity < 0.8 and len(mismatches) < 3:
                mismatches.append({
                    "index": i,
                    "expected": input_form_name,
                    "actual": item.get("form_name", ""),
                    "similarity": similarity
                })
        
        avg_score = sum(scores) / len(scores) if scores else 0
        
        return MetricResult(
            name="form_name_echo",
            score=avg_score,
            passed=avg_score >= self.thresholds["form_name_echo"],
            threshold=self.thresholds["form_name_echo"],
            details={
                "input_form": input_form_name,
                "avg_similarity": avg_score,
                "mismatches": mismatches
            },
            category="Echo-Back Consistency"
        )
    
    def check_field_name_echo(self, input_fields: List[Dict], 
                               output: List[Dict]) -> MetricResult:
        """
        2.2 Field Name Echo
        Output field names should closely match input field names.
        """
        input_names = [
            f.get("form_field_value", f.get("field_name", "")).strip()
            for f in input_fields
        ]
        input_names = [n for n in input_names if n]
        
        output_names = [
            r.get("form_field_value", "").strip()
            for r in output
        ]
        output_names = [n for n in output_names if n]
        
        if not input_names or not output_names:
            return MetricResult(
                name="field_name_echo",
                score=0,
                passed=False,
                threshold=self.thresholds["field_name_echo"],
                details={"error": "Missing input or output fields"},
                category="Echo-Back Consistency"
            )
        
        # For each output, find best matching input
        scores = []
        for out_name in output_names:
            best_match = max(
                (fuzz.ratio(out_name.lower(), inp.lower()) for inp in input_names),
                default=0
            ) / 100
            scores.append(best_match)
        
        avg_score = sum(scores) / len(scores) if scores else 0
        
        return MetricResult(
            name="field_name_echo",
            score=avg_score,
            passed=avg_score >= self.thresholds["field_name_echo"],
            threshold=self.thresholds["field_name_echo"],
            details={
                "input_count": len(input_names),
                "output_count": len(output_names),
                "avg_similarity": avg_score,
                "min_similarity": min(scores) if scores else 0,
                "max_similarity": max(scores) if scores else 0
            },
            category="Echo-Back Consistency"
        )
    
    def check_domain_validity(self, output: List[Dict]) -> MetricResult:
        """
        2.3 Domain Validity
        Domain codes should be valid CDISC domains.
        """
        if not output:
            return MetricResult(
                name="domain_validity",
                score=0,
                passed=False,
                threshold=self.thresholds["domain_validity"],
                details={"error": "Empty output"},
                category="Echo-Back Consistency"
            )
        
        valid_count = 0
        invalid_domains = []
        
        for item in output:
            domain = str(item.get("form_domain_name", "")).upper().strip()
            # Extract first 2 characters for domain code
            domain_code = domain[:2] if domain else ""
            
            if not domain_code:
                # Empty domain - count as valid (might be intentional)
                valid_count += 1
            elif domain_code in VALID_CDISC_DOMAINS:
                valid_count += 1
            else:
                if domain not in invalid_domains and len(invalid_domains) < 5:
                    invalid_domains.append(domain)
        
        score = valid_count / len(output)
        
        return MetricResult(
            name="domain_validity",
            score=score,
            passed=score >= self.thresholds["domain_validity"],
            threshold=self.thresholds["domain_validity"],
            details={
                "valid_count": valid_count,
                "total_count": len(output),
                "invalid_domains": invalid_domains,
                "valid_cdisc_domains_count": len(VALID_CDISC_DOMAINS)
            },
            category="Echo-Back Consistency"
        )

In [5]:
# Continue SelfValidatingEvaluator class - Semantic Quality metrics

class SelfValidatingEvaluator(SelfValidatingEvaluator):
    
    # ============ SEMANTIC QUALITY (LLM-as-Judge) ============
    
    def _llm_judge(self, prompt: str, max_retries: int = 2) -> float:
        """Call LLM to judge quality, returns score 0.0-1.0."""
        if not self.judge_llm:
            return 0.75  # Default neutral score if LLM not available
        
        for attempt in range(max_retries):
            try:
                # Use as_langchain_llm() which is the working pattern in this codebase
                response = self.judge_llm.invoke(prompt)
                
                # Extract numeric score from response
                text = str(response).strip()
                # Try to find a number between 0 and 1
                numbers = re.findall(r'\b(0\.\d+|1\.0|0|1)\b', text)
                if numbers:
                    return min(max(float(numbers[0]), 0.0), 1.0)
                
                # Try to find percentage
                percentages = re.findall(r'(\d+)%', text)
                if percentages:
                    return min(max(int(percentages[0]) / 100, 0.0), 1.0)
                
            except Exception as e:
                if attempt == max_retries - 1:
                    print(f"LLM Judge failed: {e}")
        
        return 0.75  # Default score on failure
    
    def check_validation_relevance(self, output: List[Dict], 
                                    sample_size: int = 5) -> MetricResult:
        """
        3.1 Validation Relevance (LLM-as-Judge)
        Does the validation logic make sense for the field type?
        """
        if not output:
            return MetricResult(
                name="validation_relevance",
                score=0,
                passed=False,
                threshold=self.thresholds["validation_relevance"],
                details={"error": "Empty output"},
                category="Semantic Quality"
            )
        
        # Sample items for LLM evaluation (to save tokens)
        samples = output[:sample_size] if len(output) > sample_size else output
        scores = []
        judgments = []
        
        for item in samples:
            field_name = item.get("form_field_value", "unknown")
            validation = item.get("validation_logic", "")
            
            if not validation:
                scores.append(0)
                continue
            
            prompt = f"""Rate if this validation rule is relevant for the field (0.0 to 1.0):

Field Name: {field_name}
Validation Rule: {validation[:200]}

Consider:
- Does the validation match the likely data type of the field?
- Is the validation meaningful for clinical data?

Return ONLY a decimal number between 0.0 and 1.0."""
            
            score = self._llm_judge(prompt)
            scores.append(score)
            judgments.append({"field": field_name, "score": score})
        
        avg_score = sum(scores) / len(scores) if scores else 0
        
        return MetricResult(
            name="validation_relevance",
            score=avg_score,
            passed=avg_score >= self.thresholds["validation_relevance"],
            threshold=self.thresholds["validation_relevance"],
            details={
                "samples_evaluated": len(samples),
                "avg_score": avg_score,
                "judgments": judgments,
                "llm_enabled": self.enable_llm_judge
            },
            category="Semantic Quality"
        )
    
    def check_logic_coherence(self, output: List[Dict],
                               sample_size: int = 5) -> MetricResult:
        """
        3.2 Logic Coherence (LLM-as-Judge)
        Is the validation logic grammatically and logically correct?
        """
        if not output:
            return MetricResult(
                name="logic_coherence",
                score=0,
                passed=False,
                threshold=self.thresholds["logic_coherence"],
                details={"error": "Empty output"},
                category="Semantic Quality"
            )
        
        samples = output[:sample_size] if len(output) > sample_size else output
        scores = []
        
        for item in samples:
            validation = item.get("validation_logic", "")
            
            if not validation:
                scores.append(0)
                continue
            
            prompt = f"""Rate the grammatical and logical correctness of this validation rule (0.0 to 1.0):

Validation Rule: {validation[:300]}

Consider:
- Is it grammatically correct?
- Is it logically coherent (not contradictory)?
- Is it clearly understandable?

Return ONLY a decimal number between 0.0 and 1.0."""
            
            score = self._llm_judge(prompt)
            scores.append(score)
        
        avg_score = sum(scores) / len(scores) if scores else 0
        
        return MetricResult(
            name="logic_coherence",
            score=avg_score,
            passed=avg_score >= self.thresholds["logic_coherence"],
            threshold=self.thresholds["logic_coherence"],
            details={
                "samples_evaluated": len(samples),
                "avg_score": avg_score,
                "llm_enabled": self.enable_llm_judge
            },
            category="Semantic Quality"
        )
    
    def check_clinical_plausibility(self, input_form_name: str,
                                     output: List[Dict],
                                     sample_size: int = 5) -> MetricResult:
        """
        3.3 Clinical Plausibility (LLM-as-Judge)
        Are the validation rules clinically reasonable?
        """
        if not output:
            return MetricResult(
                name="clinical_plausibility",
                score=0,
                passed=False,
                threshold=self.thresholds["clinical_plausibility"],
                details={"error": "Empty output"},
                category="Semantic Quality"
            )
        
        samples = output[:sample_size] if len(output) > sample_size else output
        scores = []
        
        for item in samples:
            field_name = item.get("form_field_value", "unknown")
            validation = item.get("validation_logic", "")
            
            if not validation:
                scores.append(0)
                continue
            
            prompt = f"""Rate if this clinical data validation is plausible (0.0 to 1.0):

Form: {input_form_name}
Field: {field_name}
Validation: {validation[:200]}

Consider:
- Would this validation make sense in a clinical trial context?
- Is it a reasonable data quality check?
- Would a clinical data manager expect this type of validation?

Return ONLY a decimal number between 0.0 and 1.0."""
            
            score = self._llm_judge(prompt)
            scores.append(score)
        
        avg_score = sum(scores) / len(scores) if scores else 0
        
        return MetricResult(
            name="clinical_plausibility",
            score=avg_score,
            passed=avg_score >= self.thresholds["clinical_plausibility"],
            threshold=self.thresholds["clinical_plausibility"],
            details={
                "form_name": input_form_name,
                "samples_evaluated": len(samples),
                "avg_score": avg_score,
                "llm_enabled": self.enable_llm_judge
            },
            category="Semantic Quality"
        )

In [6]:
# Continue SelfValidatingEvaluator class - Regression & Performance metrics

class SelfValidatingEvaluator(SelfValidatingEvaluator):
    
    # ============ REGRESSION & PERFORMANCE ============
    
    def check_determinism(self, input_payload: Dict,
                          first_output: List[Dict],
                          second_output: List[Dict] = None) -> MetricResult:
        """
        4.1 Determinism Score
        Same input should produce similar output (run twice if second_output not provided).
        """
        if second_output is None:
            # Skip if we only have one run
            return MetricResult(
                name="determinism_score",
                score=1.0,
                passed=True,
                threshold=self.thresholds["determinism_score"],
                details={"note": "Single run - determinism not tested"},
                category="Regression & Performance"
            )
        
        if not first_output or not second_output:
            return MetricResult(
                name="determinism_score",
                score=0,
                passed=False,
                threshold=self.thresholds["determinism_score"],
                details={"error": "Missing output for comparison"},
                category="Regression & Performance"
            )
        
        # Compare outputs
        similarities = []
        
        # Compare by field name
        first_by_field = {r.get("form_field_value", ""): r for r in first_output}
        second_by_field = {r.get("form_field_value", ""): r for r in second_output}
        
        all_fields = set(first_by_field.keys()) | set(second_by_field.keys())
        
        for field in all_fields:
            if field in first_by_field and field in second_by_field:
                logic1 = first_by_field[field].get("validation_logic", "")
                logic2 = second_by_field[field].get("validation_logic", "")
                similarity = fuzz.ratio(logic1.lower(), logic2.lower()) / 100
                similarities.append(similarity)
            else:
                similarities.append(0)  # Field missing in one output
        
        avg_score = sum(similarities) / len(similarities) if similarities else 0
        
        return MetricResult(
            name="determinism_score",
            score=avg_score,
            passed=avg_score >= self.thresholds["determinism_score"],
            threshold=self.thresholds["determinism_score"],
            details={
                "first_output_count": len(first_output),
                "second_output_count": len(second_output),
                "fields_compared": len(all_fields),
                "avg_similarity": avg_score
            },
            category="Regression & Performance"
        )
    
    def check_latency(self, duration_ms: int) -> MetricResult:
        """
        4.2 Latency Check
        Response time should be within acceptable bounds.
        """
        # Score decreases as latency increases beyond baseline
        if duration_ms <= LATENCY_BASELINE_MS:
            score = 1.0
        elif duration_ms <= LATENCY_BASELINE_MS * 1.5:
            # Linear decrease from 1.0 to threshold
            overage = (duration_ms - LATENCY_BASELINE_MS) / (LATENCY_BASELINE_MS * 0.5)
            score = 1.0 - (overage * 0.2)  # Max 20% penalty
        else:
            # More severe penalty for very slow responses
            score = max(0.5, 1.0 - (duration_ms / LATENCY_BASELINE_MS) * 0.3)
        
        return MetricResult(
            name="latency_check",
            score=score,
            passed=score >= self.thresholds["latency_check"],
            threshold=self.thresholds["latency_check"],
            details={
                "duration_ms": duration_ms,
                "baseline_ms": LATENCY_BASELINE_MS,
                "ratio": duration_ms / LATENCY_BASELINE_MS
            },
            category="Regression & Performance"
        )
    
    def check_token_efficiency(self, input_field_count: int,
                                output: List[Dict],
                                token_count: int = None) -> MetricResult:
        """
        4.3 Token Efficiency
        Output should not be excessively verbose.
        """
        if not output or input_field_count == 0:
            return MetricResult(
                name="token_efficiency",
                score=1.0,
                passed=True,
                threshold=self.thresholds["token_efficiency"],
                details={"note": "No data to evaluate"},
                category="Regression & Performance"
            )
        
        # Estimate output tokens if not provided
        if token_count is None:
            # Rough estimation: ~4 chars per token
            total_chars = sum(len(json.dumps(r)) for r in output)
            token_count = total_chars // 4
        
        tokens_per_field = token_count / input_field_count
        
        # Score based on tokens per field vs baseline
        if tokens_per_field <= TOKENS_PER_FIELD_BASELINE:
            score = 1.0
        elif tokens_per_field <= TOKENS_PER_FIELD_BASELINE * 2:
            score = 1.0 - ((tokens_per_field - TOKENS_PER_FIELD_BASELINE) / 
                          TOKENS_PER_FIELD_BASELINE) * 0.3
        else:
            score = max(0.5, 0.7 - (tokens_per_field / TOKENS_PER_FIELD_BASELINE) * 0.1)
        
        return MetricResult(
            name="token_efficiency",
            score=score,
            passed=score >= self.thresholds["token_efficiency"],
            threshold=self.thresholds["token_efficiency"],
            details={
                "estimated_tokens": token_count,
                "input_fields": input_field_count,
                "tokens_per_field": tokens_per_field,
                "baseline_per_field": TOKENS_PER_FIELD_BASELINE
            },
            category="Regression & Performance"
        )

In [11]:
# Complete SelfValidatingEvaluator class - Main evaluation method

class SelfValidatingEvaluator(SelfValidatingEvaluator):
    
    # ============ MAIN EVALUATION ============
    
    def evaluate(self, 
                 input_form_name: str,
                 input_fields: List[Dict],
                 output: List[Dict],
                 duration_ms: int = 0,
                 token_count: int = None,
                 second_output: List[Dict] = None,
                 skip_llm_judge: bool = False) -> EvaluationResult:
        """
        Run all self-validating metrics on a single generation.
        
        Args:
            input_form_name: The form name sent to LLM
            input_fields: List of field dicts sent to LLM
            output: LLM-generated validation rules
            duration_ms: Time taken for generation
            token_count: Output token count (estimated if not provided)
            second_output: Optional second run output for determinism check
            skip_llm_judge: Skip LLM-as-judge metrics (faster)
        
        Returns:
            EvaluationResult with all metrics
        """
        eval_id = str(uuid.uuid4())
        metrics = {}
        
        # Category 1: Schema & Completeness
        metrics["schema_validity"] = self.check_schema_validity(output)
        metrics["field_coverage"] = self.check_field_coverage(input_fields, output)
        metrics["non_empty_logic"] = self.check_non_empty_logic(output)
        metrics["unique_ecs_ids"] = self.check_unique_ecs_ids(output)
        
        # Category 2: Echo-Back Consistency
        metrics["form_name_echo"] = self.check_form_name_echo(input_form_name, output)
        metrics["field_name_echo"] = self.check_field_name_echo(input_fields, output)
        metrics["domain_validity"] = self.check_domain_validity(output)
        
        # Category 3: Semantic Quality (LLM-as-Judge)
        if not skip_llm_judge and self.enable_llm_judge:
            metrics["validation_relevance"] = self.check_validation_relevance(output)
            metrics["logic_coherence"] = self.check_logic_coherence(output)
            metrics["clinical_plausibility"] = self.check_clinical_plausibility(
                input_form_name, output
            )
        else:
            # Placeholder scores when LLM judge is skipped
            for name in ["validation_relevance", "logic_coherence", "clinical_plausibility"]:
                metrics[name] = MetricResult(
                    name=name,
                    score=1.0,
                    passed=True,
                    threshold=self.thresholds[name],
                    details={"note": "LLM judge skipped"},
                    category="Semantic Quality"
                )
        
        # Category 4: Regression & Performance
        metrics["determinism_score"] = self.check_determinism(
            {"form_name": input_form_name, "fields": input_fields},
            output, second_output
        )
        metrics["latency_check"] = self.check_latency(duration_ms)
        metrics["token_efficiency"] = self.check_token_efficiency(
            len(input_fields), output, token_count
        )
        
        # Calculate overall score
        scores = [m.score for m in metrics.values()]
        overall_score = sum(scores) / len(scores) if scores else 0
        all_passed = all(m.passed for m in metrics.values())
        
        result = EvaluationResult(
            eval_id=eval_id,
            timestamp=datetime.now(),
            input_form_name=input_form_name,
            input_field_count=len(input_fields),
            output_rule_count=len(output),
            metrics=metrics,
            overall_score=overall_score,
            passed=all_passed,
            duration_ms=duration_ms,
            metadata={
                "llm_judge_enabled": self.enable_llm_judge and not skip_llm_judge,
                "determinism_tested": second_output is not None
            }
        )
        
        self.results_history.append(result)
        return result
    
    def evaluate_with_generation(self,
                                  form_name: str,
                                  fields: List[Dict],
                                  test_determinism: bool = False,
                                  skip_llm_judge: bool = False,
                                  include_hallucination: bool = True) -> EvaluationResult:
        """
        Generate ECS rules and evaluate in one call.
        
        Args:
            form_name: Form name to generate rules for
            fields: Field definitions
            test_determinism: Run generation twice to test determinism
            skip_llm_judge: Skip LLM-as-judge metrics
            include_hallucination: Run hallucination metrics (default: True)
        
        Returns:
            EvaluationResult with all metrics including hallucination
        """
        start_time = time.time()
        
        try:
            # Get LLM agent
            llm_get = self.project.get_llm("agent:4NRmRXIB")
            generation_agent = llm_get.new_completion()
            
            payload = {
                "form_name": form_name,
                "field_name": fields
            }
            generation_agent.with_context({"payload": payload})
            
            # First generation
            result = generation_agent.execute()
            response_data = json.loads(result.text)
            first_output = response_data.get("response", [])
            
            # Optional second generation for determinism
            second_output = None
            if test_determinism:
                generation_agent2 = llm_get.new_completion()
                generation_agent2.with_context({"payload": payload})
                result2 = generation_agent2.execute()
                response_data2 = json.loads(result2.text)
                second_output = response_data2.get("response", [])
            
            duration_ms = int((time.time() - start_time) * 1000)
            
            # Run base evaluation
            eval_result = self.evaluate(
                input_form_name=form_name,
                input_fields=fields,
                output=first_output,
                duration_ms=duration_ms,
                second_output=second_output,
                skip_llm_judge=skip_llm_judge
            )
            
            # Add hallucination metrics if requested
            if include_hallucination and first_output:
                # Extract field names from input fields
                field_names = []
                for f in fields:
                    name = f.get("field_name") or f.get("name") or f.get("form_field_value") or ""
                    if name:
                        field_names.append(name)
                
                # Run hallucination evaluation
                hallucination_results = hallucination_evaluator.evaluate_hallucination(
                    generated=first_output,
                    input_form_name=form_name,
                    input_field_names=field_names
                )
                
                # Merge hallucination metrics into result
                eval_result.metrics.update(hallucination_results)
                
                # Recalculate overall score with hallucination metrics
                all_scores = [m.score for m in eval_result.metrics.values()]
                eval_result.overall_score = sum(all_scores) / len(all_scores) if all_scores else 0
                eval_result.passed = all(m.passed for m in eval_result.metrics.values())
                eval_result.metadata["hallucination_checked"] = True
            else:
                eval_result.metadata["hallucination_checked"] = False
            
            return eval_result
            
        except Exception as e:
            duration_ms = int((time.time() - start_time) * 1000)
            print(f"Generation failed: {e}")
            traceback.print_exc()
            
            # Return failed result
            metrics = {}
            for name, threshold in self.thresholds.items():
                metrics[name] = MetricResult(
                    name=name,
                    score=0,
                    passed=False,
                    threshold=threshold,
                    details={"error": str(e)},
                    category="Error"
                )
            
            return EvaluationResult(
                eval_id=str(uuid.uuid4()),
                timestamp=datetime.now(),
                input_form_name=form_name,
                input_field_count=len(fields),
                output_rule_count=0,
                metrics=metrics,
                overall_score=0,
                passed=False,
                duration_ms=duration_ms,
                metadata={"error": str(e)}
            )


# Initialize evaluator
evaluator = SelfValidatingEvaluator(client, project, enable_llm_judge=True)
print("\nSelfValidatingEvaluator initialized with:")
print("  - Schema & Completeness metrics (4)")
print("  - Echo-Back Consistency metrics (3)")
print("  - Semantic Quality metrics with LLM-as-Judge (3)")
print("  - Regression & Performance metrics (3)")
print("  - Hallucination metrics (2) - now integrated into evaluate_with_generation")
print(f"  - Total metrics: {len(SELF_VALIDATION_THRESHOLDS)}")

LLM Judge initialized (bedrock:AWS-Bedrock:us.anthropic.claude-sonnet-4-5-20250929-v1:0)

SelfValidatingEvaluator initialized with:
  - Schema & Completeness metrics (4)
  - Echo-Back Consistency metrics (3)
  - Semantic Quality metrics with LLM-as-Judge (3)
  - Regression & Performance metrics (3)
  - Hallucination metrics (2) - now integrated into evaluate_with_generation
  - Total metrics: 13


## 3.5 Hallucination Metrics 

In [12]:
# Hallucination Metrics - Ported from autotest_ecs.ipynb (v1)
# These extend SelfValidatingEvaluator with ground-truth-free hallucination detection

# Thresholds for hallucination metrics
HALLUCINATION_THRESHOLDS = {
    "entity_hallucination": 0.95,  # Higher = fewer hallucinations
    "score_hallucination": 0.90,
}

# Validation type patterns for score hallucination detection
VALIDATION_TYPE_PATTERNS = {
    "date": ["date", "calendar", "future", "past", "before", "after", "dtc"],
    "numeric": ["numeric", "number", "range", "greater", "less", "between", "decimal", "integer"],
    "yes_no": ["yes", "no", "valid", "y/n", "true", "false"],
    "controlled": ["controlled", "code", "terminology", "codelist", "valid value"],
    "text": ["blank", "enterable", "required", "empty", "null", "character"],
    "datetime": ["date", "time", "24-hour", "hour", "minute"]
}


class HallucinationEvaluator:
    """
    Hallucination detection metrics ported from v1 (autotest_ecs.ipynb).
    
    These metrics can run without ground truth by checking:
    1. Entity Hallucination: Generated names should echo back input names
    2. Score Hallucination: Validation logic should match field type patterns
    """
    
    def __init__(self, thresholds: Dict[str, float] = None):
        self.thresholds = thresholds or HALLUCINATION_THRESHOLDS
    
    def calculate_entity_hallucination(self, 
                                       generated: List[Dict],
                                       input_form_name: str,
                                       input_field_names: List[str]) -> MetricResult:
        """
        4.1 Entity Hallucination Detection
        
        Checks if generated output fabricates form names or field names
        that weren't in the input. Uses fuzzy matching to allow minor variations.
        
        Args:
            generated: List of generated ECS items
            input_form_name: The form name provided as input
            input_field_names: List of field names provided as input
        """
        hallucinations = []
        input_form_lower = input_form_name.lower()
        input_fields_lower = [f.lower() for f in input_field_names]
        
        for item in generated:
            # Check form name hallucination
            actual_form = str(item.get("form_name", "")).lower()
            if actual_form and fuzz.ratio(actual_form, input_form_lower) < 70:
                hallucinations.append({
                    "type": "form_name",
                    "expected": input_form_name,
                    "actual": actual_form
                })
            
            # Check field name hallucination
            actual_field = str(item.get("form_field_value", "")).lower()
            if actual_field:
                # Find best match to any input field
                best_match = max(
                    (fuzz.ratio(actual_field, f) for f in input_fields_lower),
                    default=0
                )
                if best_match < 60:
                    hallucinations.append({
                        "type": "field_name",
                        "expected_one_of": input_field_names[:3],  # Show first 3
                        "actual": actual_field
                    })
            
            # Check domain validity (should be valid CDISC domain)
            actual_domain = str(item.get("form_domain_name", "")).upper()
            if actual_domain and actual_domain not in VALID_CDISC_DOMAINS:
                hallucinations.append({
                    "type": "invalid_domain",
                    "actual": actual_domain,
                    "note": "Not a valid CDISC domain"
                })
        
        # Score: 1 - (hallucinations / total_checks)
        total_checks = len(generated) * 3  # form_name + field_name + domain per item
        score = 1 - (len(hallucinations) / total_checks) if total_checks > 0 else 1.0
        
        return MetricResult(
            name="entity_hallucination",
            score=score,
            passed=score >= self.thresholds["entity_hallucination"],
            threshold=self.thresholds["entity_hallucination"],
            details={
                "hallucination_count": len(hallucinations),
                "total_checks": total_checks,
                "hallucinations": hallucinations[:5]  # Show first 5
            },
            category="Hallucination"
        )
    
    def calculate_score_hallucination(self, 
                                      generated: List[Dict],
                                      inferred_types: List[str] = None) -> MetricResult:
        """
        4.2 Score/Validation Logic Hallucination
        
        Checks if the validation logic in generated output makes sense
        for the type of field. Uses pattern matching to infer field types.
        
        Args:
            generated: List of generated ECS items
            inferred_types: Optional list of inferred field types (date, numeric, etc.)
        """
        matches = 0
        total = len(generated)
        details_list = []
        
        for i, item in enumerate(generated):
            logic = str(item.get("validation_logic", "")).lower()
            field_name = str(item.get("form_field_value", "")).lower()
            
            # Infer type from field name if not provided
            if inferred_types and i < len(inferred_types):
                expected_type = inferred_types[i]
            else:
                expected_type = self._infer_validation_type(field_name)
            
            patterns = VALIDATION_TYPE_PATTERNS.get(expected_type, ["must", "should", "required"])
            
            if any(p in logic for p in patterns):
                matches += 1
            elif logic:
                # Partial credit for generic validation patterns
                if any(p in logic for p in ["must", "should", "cannot", "valid", "required"]):
                    matches += 0.5
                    details_list.append({
                        "field": field_name[:30],
                        "expected_type": expected_type,
                        "partial_match": True
                    })
        
        score = matches / total if total > 0 else 0.0
        
        return MetricResult(
            name="score_hallucination",
            score=score,
            passed=score >= self.thresholds["score_hallucination"],
            threshold=self.thresholds["score_hallucination"],
            details={
                "matched": matches,
                "total": total,
                "partial_matches": details_list[:5]
            },
            category="Hallucination"
        )
    
    def _infer_validation_type(self, field_name: str) -> str:
        """Infer the expected validation type from field name."""
        field_lower = field_name.lower()
        
        if any(kw in field_lower for kw in ["date", "dtc", "birth", "death"]):
            return "date"
        elif any(kw in field_lower for kw in ["age", "weight", "height", "dose", "count", "number"]):
            return "numeric"
        elif any(kw in field_lower for kw in ["sex", "gender", "race", "ethnic", "status"]):
            return "controlled"
        elif any(kw in field_lower for kw in ["time", "hour", "minute"]):
            return "datetime"
        elif any(kw in field_lower for kw in ["yes", "no", "indicator", "flag"]):
            return "yes_no"
        else:
            return "text"
    
    def evaluate_hallucination(self,
                               generated: List[Dict],
                               input_form_name: str,
                               input_field_names: List[str]) -> Dict[str, MetricResult]:
        """
        Run all hallucination metrics on generated output.
        
        Args:
            generated: List of generated ECS items
            input_form_name: The form name provided as input
            input_field_names: List of field names provided as input
            
        Returns:
            Dictionary with entity_hallucination and score_hallucination results
        """
        return {
            "entity_hallucination": self.calculate_entity_hallucination(
                generated, input_form_name, input_field_names
            ),
            "score_hallucination": self.calculate_score_hallucination(generated)
        }


# Initialize hallucination evaluator
hallucination_evaluator = HallucinationEvaluator()
print("HallucinationEvaluator initialized")
print(f"  - entity_hallucination threshold: {HALLUCINATION_THRESHOLDS['entity_hallucination']}")
print(f"  - score_hallucination threshold: {HALLUCINATION_THRESHOLDS['score_hallucination']}")


# Example usage with existing evaluator
def evaluate_with_hallucination_check(form_name: str, 
                                      field_names: List[str],
                                      generated_output: List[Dict]) -> Dict:
    """
    Combined evaluation: self-validating metrics + hallucination detection.
    
    Example:
        result = evaluate_with_hallucination_check(
            "Demographics",
            ["Date of Birth", "Sex", "Race"],
            llm_response
        )
    """
    # Run hallucination checks
    hallucination_metrics = hallucination_evaluator.evaluate_hallucination(
        generated_output, form_name, field_names
    )
    
    # Calculate combined score
    scores = [m.score for m in hallucination_metrics.values()]
    avg_score = sum(scores) / len(scores) if scores else 0
    all_passed = all(m.passed for m in hallucination_metrics.values())
    
    return {
        "metrics": hallucination_metrics,
        "hallucination_score": avg_score,
        "passed": all_passed,
        "summary": {
            "entity_hallucination": hallucination_metrics["entity_hallucination"].score,
            "score_hallucination": hallucination_metrics["score_hallucination"].score
        }
    }

print("\nUsage: evaluate_with_hallucination_check(form_name, field_names, generated_output)")

HallucinationEvaluator initialized
  - entity_hallucination threshold: 0.95
  - score_hallucination threshold: 0.9

Usage: evaluate_with_hallucination_check(form_name, field_names, generated_output)


## 4. Test Cases

In [13]:
def create_test_input(form_name: str, field_names: List[str]) -> Tuple[str, List[Dict]]:
    """
    Create test input from just form name and field names.
    No ground truth required!
    """
    fields = [
        {
            "ecs_id": f"{form_name[:2].upper()}{i+1:03d}",
            "form_field_value": field_name
        }
        for i, field_name in enumerate(field_names)
    ]
    return form_name, fields


# Example test cases - can be any form/fields, no expected values needed!
SAMPLE_TEST_CASES = [
    create_test_input(
        "Vital Signs",
        ["Systolic Blood Pressure", "Diastolic Blood Pressure", "Heart Rate", 
         "Temperature", "Respiratory Rate", "Assessment Date"]
    ),
    create_test_input(
        "Demographics",
        ["Date of Birth", "Sex", "Race", "Ethnicity", "Country"]
    ),
    create_test_input(
        "Adverse Events",
        ["AE Start Date", "AE End Date", "AE Term", "Severity", 
         "Relationship to Study Drug", "Outcome", "Serious"]
    ),
    create_test_input(
        "Medical History",
        ["Condition", "Start Date", "End Date", "Ongoing", "Comments"]
    ),
    create_test_input(
        "Concomitant Medications",
        ["Medication Name", "Indication", "Start Date", "End Date", 
         "Dose", "Unit", "Frequency", "Route"]
    ),
]

print(f"Created {len(SAMPLE_TEST_CASES)} sample test cases")
for form_name, fields in SAMPLE_TEST_CASES:
    print(f"  - {form_name}: {len(fields)} fields")

Created 5 sample test cases
  - Vital Signs: 6 fields
  - Demographics: 5 fields
  - Adverse Events: 7 fields
  - Medical History: 5 fields
  - Concomitant Medications: 8 fields


## 5. Run Tests

In [14]:
# Run all sample test cases
all_results = []

print("="*60)
print("RUNNING SELF-VALIDATING TESTS")
print("="*60)

for i, (form_name, fields) in enumerate(SAMPLE_TEST_CASES, 1):
    print(f"\n[{i}/{len(SAMPLE_TEST_CASES)}] Testing: {form_name}")
    print(f"  Fields: {len(fields)}")
    print("-" * 40)
    
    # Run evaluation with generation
    result = evaluator.evaluate_with_generation(
        form_name=form_name,
        fields=fields,
        test_determinism=False,  # Set True to test consistency
        skip_llm_judge=False     # Set True for faster runs
    )
    
    all_results.append(result)
    
    # Print summary
    status = "PASS" if result.passed else "FAIL"
    print(f"  Status: {status}")
    print(f"  Overall Score: {result.overall_score:.1%}")
    print(f"  Duration: {result.duration_ms}ms")
    print(f"  Rules Generated: {result.output_rule_count}")
    
    # Print metrics by category
    categories = {}
    for name, metric in result.metrics.items():
        cat = metric.category or "Other"
        if cat not in categories:
            categories[cat] = []
        categories[cat].append(metric)
    
    for cat, metrics in categories.items():
        print(f"\n  {cat}:")
        for m in metrics:
            icon = "v" if m.passed else "x"
            print(f"    [{icon}] {m.name}: {m.score:.1%} (>= {m.threshold:.0%})")

print("\n" + "=" * 60)
print("TESTING COMPLETE")
print("=" * 60)

RUNNING SELF-VALIDATING TESTS

[1/5] Testing: Vital Signs
  Fields: 6
----------------------------------------
  Status: FAIL
  Overall Score: 95.8%
  Duration: 55526ms
  Rules Generated: 6

  Schema & Completeness:
    [v] schema_validity: 100.0% (>= 95%)
    [v] field_coverage: 100.0% (>= 90%)
    [v] non_empty_logic: 100.0% (>= 95%)
    [v] unique_ecs_ids: 100.0% (>= 100%)

  Echo-Back Consistency:
    [v] form_name_echo: 100.0% (>= 80%)
    [v] field_name_echo: 100.0% (>= 75%)
    [v] domain_validity: 100.0% (>= 90%)

  Semantic Quality:
    [v] validation_relevance: 100.0% (>= 70%)
    [v] logic_coherence: 95.0% (>= 80%)
    [v] clinical_plausibility: 93.0% (>= 70%)

  Regression & Performance:
    [v] determinism_score: 100.0% (>= 80%)
    [x] latency_check: 50.0% (>= 90%)
    [v] token_efficiency: 99.5% (>= 90%)

  Hallucination:
    [v] entity_hallucination: 100.0% (>= 95%)
    [v] score_hallucination: 100.0% (>= 90%)

[2/5] Testing: Demographics
  Fields: 5
-------------------

## 6. Results Summary

In [23]:
# Generate summary report
def generate_summary(results: List[EvaluationResult]) -> pd.DataFrame:
    """Generate summary DataFrame from evaluation results."""
    rows = []
    
    for r in results:
        row = {
            "form_name": r.input_form_name,
            "field_count": r.input_field_count,
            "rule_count": r.output_rule_count,
            "overall_score": r.overall_score,
            "passed": r.passed,
            "duration_ms": r.duration_ms
        }
        
        # Add individual metrics
        for name, metric in r.metrics.items():
            row[name] = metric.score
        
        rows.append(row)
    
    return pd.DataFrame(rows)


summary_df = generate_summary(all_results)

print("=" * 60)
print("SUMMARY REPORT")
print("=" * 60)

# Overall stats
passed_count = sum(1 for r in all_results if r.passed)
total_count = len(all_results)
avg_score = sum(r.overall_score for r in all_results) / total_count if total_count else 0

print(f"Tests Passed: {passed_count}/{total_count} ({passed_count/total_count:.0%})")
print(f"Average Score: {avg_score:.1%}")
print(f"Average Duration: {summary_df['duration_ms'].mean():.0f}ms")

# Metric averages by category
print("Metric Averages by Category:")

# Define metric categories
METRIC_CATEGORIES = {
    "Schema & Completeness": ["schema_validity", "field_coverage", "non_empty_logic", "unique_ecs_ids"],
    "Echo-Back Consistency": ["form_name_echo", "field_name_echo", "domain_validity"],
    "Semantic Quality (LLM)": ["validation_relevance", "logic_coherence", "clinical_plausibility"],
    "Regression & Performance": ["determinism_score", "latency_check", "token_efficiency"],
    "Hallucination": ["entity_hallucination", "score_hallucination"]
}

for category, metrics in METRIC_CATEGORIES.items():
    print(f"{category}:")
    category_present = False
    for col in metrics:
        if col in summary_df.columns:
            category_present = True
            avg = summary_df[col].mean()
            threshold = SELF_VALIDATION_THRESHOLDS.get(col, 0)
            status = "OK" if avg >= threshold else "LOW"
            print(f"    {col}: {avg:.1%} (threshold: {threshold:.0%}) [{status}]")
        else:
            print(f"    {col}: NOT EVALUATED")
    
    if not category_present:
        print(f"    (No metrics evaluated for this category)")

# Hallucination summary section
print("-" * 60)
print("HALLUCINATION DETECTION SUMMARY")
print("-" * 60)

hallucination_metrics = ["entity_hallucination", "score_hallucination"]
hallucination_present = any(m in summary_df.columns for m in hallucination_metrics)

if hallucination_present:
    for metric in hallucination_metrics:
        if metric in summary_df.columns:
            avg = summary_df[metric].mean()
            threshold = SELF_VALIDATION_THRESHOLDS.get(metric, HALLUCINATION_THRESHOLDS.get(metric, 0))
            passed_pct = (summary_df[metric] >= threshold).mean() * 100
            status = "PASS" if avg >= threshold else "FAIL"
            print(f"  {metric}:")
            print(f"    Average Score: {avg:.1%}")
            print(f"    Threshold: {threshold:.0%}")
            print(f"    Tests Passed: {passed_pct:.0f}%")
            print(f"    Status: [{status}]")
        else:
            print(f"  {metric}: NOT EVALUATED")
    
    # Calculate overall hallucination score
    hal_cols = [m for m in hallucination_metrics if m in summary_df.columns]
    if hal_cols:
        overall_hal_score = summary_df[hal_cols].mean().mean()
        print(f"Overall Hallucination Score: {overall_hal_score:.1%}")
        if overall_hal_score >= 0.90:
            print("  Verdict: LOW HALLUCINATION RISK")
        elif overall_hal_score >= 0.70:
            print("  Verdict: MODERATE HALLUCINATION RISK")
        else:
            print("  Verdict: HIGH HALLUCINATION RISK - Review generated outputs")
else:
    print("  WARNING: Hallucination metrics were NOT evaluated!")
    print("  Possible reasons:")
    print("    - LLM output was empty")
    print("    - Generation errors occurred")
    print("    - include_hallucination=False was passed")
    print("  To enable hallucination checking, ensure:")
    print("    1. LLM generates non-empty output")
    print("    2. Use evaluate_with_generation() or validate_ecs_generation()")
    print("    3. Set include_hallucination=True (default)")

print("=" * 60)

# Display DataFrame
display(summary_df.round(3))

SUMMARY REPORT
Tests Passed: 1/5 (20%)
Average Score: 95.2%
Average Duration: 62396ms
Metric Averages by Category:
Schema & Completeness:
    schema_validity: 100.0% (threshold: 95%) [OK]
    field_coverage: 100.0% (threshold: 90%) [OK]
    non_empty_logic: 96.0% (threshold: 95%) [OK]
    unique_ecs_ids: 100.0% (threshold: 100%) [OK]
Echo-Back Consistency:
    form_name_echo: 100.0% (threshold: 80%) [OK]
    field_name_echo: 100.0% (threshold: 75%) [OK]
    domain_validity: 100.0% (threshold: 90%) [OK]
Semantic Quality (LLM):
    validation_relevance: 94.6% (threshold: 70%) [OK]
    logic_coherence: 87.0% (threshold: 80%) [OK]
    clinical_plausibility: 88.4% (threshold: 70%) [OK]
Regression & Performance:
    determinism_score: 100.0% (threshold: 80%) [OK]
    latency_check: 70.0% (threshold: 90%) [LOW]
    token_efficiency: 98.7% (threshold: 90%) [OK]
Hallucination:
    entity_hallucination: 100.0% (threshold: 0%) [OK]
    score_hallucination: 94.0% (threshold: 0%) [OK]
-------------

,form_name,field_count,rule_count,overall_score,passed,duration_ms,schema_validity,field_coverage,non_empty_logic,unique_ecs_ids,form_name_echo,field_name_echo,domain_validity,validation_relevance,logic_coherence,clinical_plausibility,determinism_score,latency_check,token_efficiency,entity_hallucination,score_hallucination
0,Vital Signs,6,6,0.958,False,55526,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.00,0.95,0.93,1.0,0.5,0.995,1.0,1.0
1,Demographics,5,5,0.974,True,13722,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.99,0.83,0.90,1.0,1.0,0.984,1.0,0.9
2,Adverse Events,7,7,0.959,False,131077,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.00,0.95,0.96,1.0,0.5,0.970,1.0,1.0
3,Medical History,5,5,0.919,False,14269,1.0,1.0,0.8,1.0,1.0,1.0,1.0,0.75,0.70,0.74,1.0,1.0,0.988,1.0,0.8
4,Concomitant Medications,8,8,0.953,False,97387,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.99,0.92,0.89,1.0,0.5,1.000,1.0,1.0


## 7. SCHEMA & COMPLETENESS METRICS (4 metrics)

These metrics verify that LLM output has correct structure and covers all inputs.

| Metric | Description |
|--------|-------------|
| **Schema Validity** | Percentage of output rules that contain all required fields |
| **Field Coverage** | Percentage of input fields that have corresponding output rules |
| **Non-Empty Logic** | Percentage of rules with meaningful validation logic (not empty/trivial) **Detection Logic:** Validation text must be > 10 characters, Cannot be just "N/A", "None", "TBD", or empty|
| **Unique ECS IDs** | Percentage of ECS IDs that are unique (no duplicates) |

## 8. ECHO-BACK CONSISTENCY METRICS (3 metrics)

These metrics verify that LLM output faithfully reflects the input data.

| Metric | Description |
|--------|-------------|
| **Form Name Echo** | Fuzzy similarity between input form name and output form names |
| **Field Name Echo** | Best-match fuzzy similarity between output and input field names |
| **Domain Validity** | Percentage of rules with valid CDISC domain codes |

## 9. SEMANTIC QUALITY METRICS - LLM-as-Judge (3 metrics)

These metrics use a secondary LLM to evaluate the quality of generated rules.

| Metric | Description |
|--------|-------------|
| **Validation Relevance** | Whether validation logic is appropriate for the field type |
| **Logic Coherence** | Whether validation logic is grammatically and logically correct |
| **Clinical Plausibility** | Whether validation rules are clinically reasonable |

## 10. REGRESSION & PERFORMANCE METRICS (3 metrics)

These metrics monitor consistency and efficiency over time.

| Metric | Description |
|--------|-------------|
| **Determinism Score** | Similarity of outputs when same input is run twice |
| **Latency Check** | Whether response time is within acceptable bounds |
| **Token Efficiency** | Whether output is not excessively verbose **Baseline:** `TOKENS_PER_FIELD_BASELINE = 500`|

## 11 Hallucination Metrics 

These metrics to detect hallucinations in generated ECS.

| Metric | Description |
|--------|-------------|
| `entity_hallucination` | Checks if form names and domains match input (not fabricated) |
| `score_hallucination` | Validates that logic patterns match expected field types |